In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# regression modules
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# classification modules
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

In [7]:
df = pd.read_csv('data/processed/spotify_data_cleaned.csv')

# dropping the idenifier columns since no prediction can be made on them.

df = df.drop(columns=["track_id", "artists", "album_name", "track_name"], errors = 'ignore')

# encoding 'explicit' as binary integer, 1-true, 0-false
df['explicit'] = df['explicit'].astype(int)

TARGET = 'popularity'

NUMERIC_FEATURES = [
    "danceability", "energy", "loudness", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "tempo", "duration_min",
    "explicit", "mode"
]

OHE_INT_FEATURES = ["key", "time_signature"]
OHE_STR_FEATURES = ["track_genre"]

x = df[NUMERIC_FEATURES + OHE_INT_FEATURES + OHE_STR_FEATURES]
y = df[TARGET]

print(f"Dataset: {x.shape[0]:,} songs with {x.shape[1]:,} features each.")
print(f"Popularity - min: {y.min()}, max: {y.max()}, mean: {y.mean():.2f}, median: {y.median()}, std: {y.std():.2f}")


Dataset: 97,980 songs with 15 features each.
Popularity - min: 1, max: 100, mean: 38.67, median: 39.0, std: 19.20


In [8]:
# preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers = [
        ('num', StandardScaler(), NUMERIC_FEATURES),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), OHE_INT_FEATURES),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False), OHE_STR_FEATURES)
    ], remainder='drop'
)

print("Preprocessing pipeline created successfully.")
print(f"  • Scaled numeric : {len(NUMERIC_FEATURES)} features")
print(f"  • OHE string     : {OHE_STR_FEATURES}  (114 genres → 114 columns)")
print(f"  • OHE integer    : {OHE_INT_FEATURES}   (key × 12 + time_sig × 5)")

Preprocessing pipeline created successfully.
  • Scaled numeric : 12 features
  • OHE string     : ['track_genre']  (114 genres → 114 columns)
  • OHE integer    : ['key', 'time_signature']   (key × 12 + time_sig × 5)


In [12]:
# regression, train/test split.

y_log = np.log1p(y)

y_bins = pd.cut(y, bins = [0,30,60,100], labels = ['Low', 'Medium', 'High'])

x_train, x_test, y_train_raw, y_test_raw = train_test_split(x, y, test_size = 0.2, random_state = 42, stratify = y_bins)

y_train_log = np.log1p(y_train_raw)
y_test_log = np.log1p(y_test_raw)


print(f"Train: {len(x_train):,} | Test: {len(x_test):,}")
print(f"\nTier distribution in train set:")
print(pd.cut(y_train_raw, bins= [0,30,60,100], labels = ["Low", "Medium", "High"]).value_counts(normalize= True).sort_index().round(3))


Train: 78,384 | Test: 19,596

Tier distribution in train set:
popularity
Low       0.365
Medium    0.497
High      0.138
Name: proportion, dtype: float64
